# Generate the dataset

- Extract paragraphs (chunks) from a corpus of construction-related documents in PDF format and save them to a JSON file.

- Ask OpenAI to generate (synthesize) a query that is suitable to answer each paragraph.

- Save the query-paragraph pairs to a JSON file.

- Remove duplicated paragraphs since they may later induce data leakage.

## 1. Extract paragraphs

### 1.1. Install libraries

We use [PyMuPDF4LLM](https://pymupdf.readthedocs.io/en/latest/pymupdf4llm/index.html) for parsing PDF documents.

In [ ]:
%pip install pymupdf4llm

### 1.2. Import libraries

In [ ]:
import json
import re
from datetime import datetime
from pathlib import Path

import pymupdf4llm
from google.colab import drive

### 1.3. Functions for paragraph extraction

In [ ]:
MIN_PAR_WORDS = 40
MAX_PAR_WORDS = 300
MIN_ALPHA_RATIO = 0.65
MAX_PIPE_RATION = 0.05
MIN_SENTENCE_MARKERS = 2

def clean_markdown(md):

    """Clean and normalize the Markdown obtained from a PDF document.

    This function:
    - Removes image placeholder blocks inserted by PyMuPDF4LLM.
    - Converts HTML `<br>` tags into Markdown paragraph breaks.
    - Collapses multiple consecutive blank lines into a maximum of two.
    - Strips leading and trailing whitespace.
    """

    # Remove image text blocks inserted by PyMuPDF4LLM
    md = re.sub(
        r"\*\*----- Start of .*?-----\*\*.*?\*\*----- End of .*?-----\*\*",
        "",
        md,
        flags=re.DOTALL,
    )

    # Normalize HTML line breaks into markdown paragraph breaks
    md = md.replace("<br>", "\n\n")

    # Clean excessive blank lines (but preserve structure)
    md = re.sub(r"\n{3,}", "\n\n", md)

    return md.strip()


def seems_a_list_of_names(text):
    """
    Return whether a paragraph looks like a list of words
    """
    words = text.lower().split()

    # Lists of names don't have infinitives, nor adverbs,
    # nor sustantives derived from verbs.
    not_a_list_suffixes = ("ar", "er", "ir", "ción", "mente")

    sentence_markers = sum(w.endswith(not_a_list_suffixes) for w in words)

    return sentence_markers < MIN_SENTENCE_MARKERS


def seems_a_table(text):
    """
    Return whether a paragraph looks like a table
     """
    # Tables contain too many pipe characters
    pipe_ratio = text.count("|") / max(len(text), 1)

    if pipe_ratio > MAX_PIPE_RATION:
        return True


def is_valid_paragraph(paragraph,
              min_par_words=MIN_PAR_WORDS,
              max_par_words=MAX_PAR_WORDS,
              min_alpha_ratio=MIN_ALPHA_RATIO):
    """
    Return whether a paragraph meets the minimum quality requirements.

    A paragraph is regarded as not valid when:
    - It is too short
    - It is too  long
    - It has a small alpha ratio
    - It seems to be list of names
    - It seems to be a table
    """

    word_count = len(paragraph.split())
    if word_count < min_par_words:
        return False

    if word_count > max_par_words:
        return False

    alpha_ratio = sum(c.isalpha() for c in paragraph) / max(len(paragraph), 1)
    if alpha_ratio < min_alpha_ratio:
        return False

    if seems_a_list_of_names(paragraph):
        return False

    if seems_a_table(paragraph):
        return False

    return True

### 1.4 Extract paragraphs and save them to a JSON file

In [ ]:
# Mount Google Drive for file access
drive.mount("/content/drive", force_remount=False)

#  Document corpus and dataset directories
PDF_DIR = "/content/drive/MyDrive/RAG_UPC_Final_project/construction_related_documents"
DATASET_DIR = PDF_DIR

# JSON file full path
tst = str(int(datetime.now().timestamp()))
json_filename = f"dataset_paragraphs_{tst}.json"
json_file_path = f"{DATASET_DIR}/{json_filename}"

# Parse document corpus and extract valid paragraphs
i = 1
paragraphs = []
dataset = []
pdf_paths = list(Path(PDF_DIR).glob("*.pdf"))
for pdf_path in pdf_paths:
    doc_id = f"{pdf_path.stem}.pdf"
    print(f"parsing {doc_id} ...")
    try:
        md = pymupdf4llm.to_markdown(pdf_path, use_ocr=False)
    except Exception as e:
        print(f"Error in {pdf_path.name}: {e}")
        continue
    md_clean = clean_markdown(md)

    # In Markdown the paragraph mark is '\n\n'
    for p in md_clean.split("\n\n"):
         if is_valid_paragraph(p):
             dataset.append({
                "id": i,
                "doc_id": doc_id,
                "query": "",
                "chunk": p,
             })
             i += 1

# Save extracted paragraphs to a JSON file
with open(json_file_path, "w", encoding="utf-8") as f:
    json.dump(dataset, f, indent=4, ensure_ascii=False)

# End
print()
print("End of paragraph extraction")

## 2. Synthesize queries

### 2.1. Install libraries
We use [openai](https://pypi.org/project/openai/) for asking OpenAI to synthesize the queries that paragraphs provides relevant context for.

In [ ]:
%pip install openai

### 2.2. Import libraries

In [ ]:
import json
import time
from datetime import datetime
from getpass import getpass

from google.colab import drive
from openai import OpenAI

### 2.3. Functions for synthesizing queries

In [ ]:
# Set the OpenAI model, prompt for API key, and initialize the OpenAI client
OPENAI_MODEL = "gpt-4.1-mini"
openai_api_key = getpass("Enter your OpenAI api key: ")
client = OpenAI(api_key=openai_api_key)

def generate_query(chunk: str, openai_model=OPENAI_MODEL) -> str:
    """
    Generates a Spanish user query for a text chunk using an OpenAI model.
    """

    prompt = f"""
    Eres un experto en recuperación de información para sistemas RAG.

    Las preguntas generadas se utilizarán para entrenar un reranker. Su objetivo es ayudar a distinguir el fragmento correcto entre otros similares.

    Dado un fragmento de texto en español relativo a la construcción y otros temas afines, genera una única pregunta natural de usuario en español que pueda ser respondida completamente por el texto.

    Reglas:
    - La respuesta debe encontrarse íntegramente en el fragmento.
    - La pregunta debe ser específica y discriminativa.
    - No copies frases exactas del texto.
    - No menciones términos técnicos innecesarios.
    - Debe sonar como una consulta real de búsqueda.
    - Evita preguntas demasiado generales.
    - Devuelve solo la pregunta.

    Texto:
    {chunk}
     """

    response = client.chat.completions.create(
        model=openai_model,
        messages=[
            {"role": "user", "content": prompt}
        ],
        temperature=0.7
    )

    return response.choices[0].message.content.strip()

### 2.4 Synthesize the queries and save the query-paragraph pairs to a JSON file

In [ ]:
# Mount Google Drive for file access
drive.mount("/content/drive", force_remount=False)

#  Dataset directory
DATASET_DIR = "/content/drive/MyDrive/RAG_UPC_Final_project/construction_related_documents"

# JSON files paths
data_file_name_in = "dataset_paragraphs_1782414229.json" # @param {type:"string"}
data_file_path_in = f"{DATASET_DIR}/{data_file_name_in}"
tst = str(int(datetime.now().timestamp()))
data_file_name_out = f"dataset_queries_paragraphs_{tst}.json"
data_file_path_out = f"{DATASET_DIR}/{data_file_name_out}"


# Load the file with the paragraphs
with open(data_file_path_in, "r", encoding="utf-8") as f:
    data = json.load(f)

# Obtain a suitable query for each paragraph
for i, item in enumerate(data):

    if item.get("query"):
        continue

    try:
        item["query"] = generate_query(item["chunk"])
        print(f"[{i+1}] OK")

        time.sleep(0.2)  # rate limit control

    except Exception as e:
        print(f"[{i+1}] ERROR: {e}")
        item["query"] = ""

# Save query-paragraph pairs to a file
with open(data_file_path_out, "w", encoding="utf-8") as f:
    json.dump(data, f, ensure_ascii=False, indent=2)

# End
print("End of query generation")

### 2.5 Remove chunk duplicates since they induce data leakage

In [ ]:
import json
from google.colab import drive

# Mount Google Drive for file access
drive.mount("/content/drive", force_remount=False)

#  Dataset directory
DATASET_DIR = "/content/drive/MyDrive/RAG_UPC_Final_project/construction_related_documents"

# JSON files paths
data_file_name_in = "dataset_queries_paragraphs_1782488748.json" # @param {type:"string"}
data_file_path_in = f"{DATASET_DIR}/{data_file_name_in}"
tst = str(int(datetime.now().timestamp()))
data_file_name_out = f"dataset_queries_paragraphs_{tst}.json"
data_file_path_out = f"{DATASET_DIR}/{data_file_name_out}"

# Read the dataset
with open(data_file_path_in, "r", encoding="utf-8") as f:
    data = json.load(f)

seen_chunks = set()
clean_data = []

total_records = len(data)
removed_records = 0

for record in data:
    chunk = record["chunk"].strip()

    if chunk not in seen_chunks:
        seen_chunks.add(chunk)
        clean_data.append(record)
    else:
        removed_records += 1

# Save the purged dataset
with open(data_file_path_out, "w", encoding="utf-8") as f:
    json.dump(clean_data, f, ensure_ascii=False, indent=2)

print(f"Registros originales: {total_records}")
print(f"Registros eliminados: {removed_records}")
print(f"Registros finales: {len(clean_data)}")
print(f"Fichero generado: {data_file_path_out}")